In [ ]:
!pip install -q opencv-python tqdm matplotlib scikit-learn

import os, sys, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms, models
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
CONFIG = {
    'dataset_root': '/kaggle/input/celeba-spoof-for-face-antispoofing/CelebA Spoof For Face AntiSpoofing/CelebA_Spoof_/CelebA_Spoof',
    'output_dir': '/kaggle/working/context_mobilenetv2_224',
    'cropped_dir': '/kaggle/working/celeba_spoof_context_224',
    'image_size': 224,
    'context_margin_ratio': 1.2,
    'batch_size': 64,
    'epochs': 8,
    'patience': 3,
    'resume': False,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'val_ratio': 0.1,
    'test_ratio': 0.1,
}
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['cropped_dir'], exist_ok=True)
CONFIG


In [ ]:
# ===== SMOKE DEBUG: verify dataset paths exist before full prep =====
# If you get errors below, the dataset slug / folder structure differs from expected.
# Common causes:
#   - You attached a different CelebA-Spoof dataset (different slug)
#   - Folder layout differs (e.g. spaces in name escaped or removed)
# Adjust CONFIG['dataset_root'] above and re-run this cell.

ROOT_DEBUG = Path(CONFIG['dataset_root'])
print('dataset_root:', ROOT_DEBUG)
print('exists:', ROOT_DEBUG.exists())
print()

if not ROOT_DEBUG.exists():
    print('PATH NOT FOUND — listing /kaggle/input/:')
    !ls /kaggle/input/
    print()
    print('Deep search for CelebA Spoof folder:')
    !find /kaggle/input -maxdepth 6 -type d -iname '*celeba*spoof*' 2>/dev/null | head -20
    raise FileNotFoundError(f'Adjust CONFIG[\'dataset_root\'] to match what was printed above, then re-run.')

print('Children of dataset_root:')
!ls "{ROOT_DEBUG}"
print()
print('metas/intra_test contents:')
!ls "{ROOT_DEBUG}/metas/intra_test" 2>/dev/null
print()
print('Sample Data/train subdirs:')
!ls "{ROOT_DEBUG}/Data/train" 2>/dev/null | head -10
print()
# Pick first ID dir and show one image + its _BB.txt
import subprocess
first_id_dir = subprocess.getoutput(f'ls "{ROOT_DEBUG}/Data/train" | head -1').strip()
print(f'First train ID dir: {first_id_dir}')
print('Contents of first ID dir / live (or spoof):')
!ls "{ROOT_DEBUG}/Data/train/{first_id_dir}/live" 2>/dev/null | head -10
!ls "{ROOT_DEBUG}/Data/train/{first_id_dir}/spoof" 2>/dev/null | head -10


In [ ]:
ROOT = Path(CONFIG['dataset_root'])
train_label_dir = ROOT / 'metas' / 'intra_test'

def load_split(label_file: Path) -> pd.DataFrame:
    rows = []
    with open(label_file) as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            parts = line.split()
            img_rel, label = parts[0], int(parts[1])
            rows.append({'image_path': img_rel, 'label': 1 if label == 0 else 0})
            # CelebA-Spoof: label 0 = live; label 1+ = spoof. We invert to: 1=live, 0=spoof.
    return pd.DataFrame(rows)

train_df = load_split(train_label_dir / 'train_label.txt')
test_df = load_split(train_label_dir / 'test_label.txt')
print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print('Train label distribution:', train_df['label'].value_counts().to_dict())

In [ ]:
def show_raw_samples(df, n_per_class=3):
    fig, axes = plt.subplots(2, n_per_class, figsize=(4*n_per_class, 8))
    for cls, label_name in enumerate(['spoof', 'live']):
        samples = df[df['label'] == cls].sample(n_per_class, random_state=SEED)
        for i, (_, row) in enumerate(samples.iterrows()):
            img = cv2.imread(str(ROOT / row['image_path']))
            if img is None:
                axes[cls, i].set_title(f'{label_name} (missing)')
                continue
            axes[cls, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[cls, i].set_title(f'{label_name}\n{row["image_path"][-30:]}')
            axes[cls, i].axis('off')
    plt.tight_layout()
    plt.savefig(Path(CONFIG['output_dir']) / 'viz1_raw_samples.png', dpi=80)
    plt.show()

show_raw_samples(train_df)

In [ ]:
def load_bbox_for_image(image_rel: str, image_shape: tuple[int, int]) -> tuple[int, int, int, int] | None:
    """Return (x1, y1, x2, y2) bbox in ORIGINAL image pixel coordinates.

    CelebA-Spoof convention: <basename>_BB.txt contains "x y w h confidence"
    where coordinates are relative to a 224x224 reference frame used during
    annotation. They MUST be rescaled to original image dimensions.

    image_shape: (H, W) of the original image — used for rescaling.
    """
    image_path = Path(image_rel)
    bb_name = image_path.stem + '_BB.txt'
    bb_path = ROOT / image_path.parent / bb_name
    if not bb_path.exists():
        return None
    try:
        line = bb_path.read_text().strip().split('\n')[0]
        parts = line.split()
        if len(parts) < 4:
            return None
        x224, y224, w224, h224 = map(float, parts[:4])
        if w224 <= 0 or h224 <= 0:
            return None
        orig_h, orig_w = image_shape[0], image_shape[1]
        sx = orig_w / 224.0
        sy = orig_h / 224.0
        x1 = int(round(x224 * sx))
        y1 = int(round(y224 * sy))
        x2 = int(round((x224 + w224) * sx))
        y2 = int(round((y224 + h224) * sy))
        # Clamp to image bounds
        x1 = max(0, min(x1, orig_w - 1))
        y1 = max(0, min(y1, orig_h - 1))
        x2 = max(x1 + 1, min(x2, orig_w))
        y2 = max(y1 + 1, min(y2, orig_h))
        return (x1, y1, x2, y2)
    except Exception:
        return None


# Sanity check: load 5 samples, print bbox + verify it covers a meaningful portion of the image
print('Bbox sanity check on first 5 train samples (after 224->original rescale):')
ok_count = 0
for idx in range(min(5, len(train_df))):
    rel = train_df.iloc[idx]['image_path']
    img = cv2.imread(str(ROOT / rel))
    if img is None:
        print(f'  [{idx}] {rel}: image MISSING')
        continue
    bb = load_bbox_for_image(rel, img.shape)
    if bb is None:
        print(f'  [{idx}] {rel}: bbox MISSING')
        continue
    x1, y1, x2, y2 = bb
    h, w = img.shape[:2]
    bbox_area_frac = ((x2-x1) * (y2-y1)) / (w * h)
    print(f'  [{idx}] {rel}')
    print(f'        img {w}x{h}, bbox {bb}, area_frac={bbox_area_frac:.3f}')
    if 0.02 < bbox_area_frac < 0.95:
        ok_count += 1

assert ok_count >= 3, f'Bbox rescale looks wrong (only {ok_count}/5 plausible). Inspect raw _BB.txt values.'
print(f'\nOK: {ok_count}/5 bboxes have plausible area coverage. Bbox loader looks correct.')


In [ ]:
# ===== Disk-space pre-flight (skip silently if not on Kaggle) =====
import shutil

cropped_path = Path(CONFIG['cropped_dir'])
total, used, free = shutil.disk_usage(cropped_path.parent if cropped_path.parent.exists() else '/')
free_gb = free / (1024 ** 3)
print(f'Free space at {cropped_path.parent}: {free_gb:.1f} GB')

# Rough estimate: full CelebA-Spoof = ~625K images, ~18KB per JPEG q75 224x224 = ~11 GB
# With limit=None, ensure at least 13 GB free for safety margin.
estimated_gb = 13.0 if not any('limit' in str(line) and 'limit=None' for line in []) else 13.0
if free_gb < estimated_gb:
    print(f'WARNING: only {free_gb:.1f} GB free; full prep estimated to need ~{estimated_gb} GB.')
    print('Consider lowering JPEG quality further (50 instead of 75) or limiting dataset size.')
else:
    print(f'OK: enough headroom for full prep at JPEG quality 75 (~11 GB estimated).')


In [ ]:
def expand_and_pad_crop(image_bgr, bbox_xyxy, margin=0.8, out_size=224):
    h, w = image_bgr.shape[:2]
    x1, y1, x2, y2 = bbox_xyxy
    bw, bh = x2 - x1, y2 - y1
    mx, my = int(bw * margin), int(bh * margin)
    ex1, ey1 = max(0, x1 - mx), max(0, y1 - my)
    ex2, ey2 = min(w, x2 + mx), min(h, y2 + my)
    crop = image_bgr[ey1:ey2, ex1:ex2]
    # Pad to square
    ch, cw = crop.shape[:2]
    side = max(ch, cw)
    pt, pl = (side - ch) // 2, (side - cw) // 2
    pb, pr = side - ch - pt, side - cw - pl
    mean_bgr = crop.reshape(-1, 3).mean(axis=0).astype(int).tolist()
    padded = cv2.copyMakeBorder(crop, pt, pb, pl, pr, cv2.BORDER_CONSTANT, value=mean_bgr)
    return cv2.resize(padded, (out_size, out_size), interpolation=cv2.INTER_LINEAR)


def process_split(df, split_name, limit=None):
    out_dir = Path(CONFIG['cropped_dir']) / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    iterator = df.head(limit).iterrows() if limit else df.iterrows()
    for i, row in tqdm(iterator, total=limit or len(df), desc=split_name):
        img = cv2.imread(str(ROOT / row['image_path']))
        if img is None: continue
        bbox = load_bbox_for_image(row['image_path'], img.shape)
        if bbox is None: continue
        cropped = expand_and_pad_crop(img, bbox, CONFIG['context_margin_ratio'], CONFIG['image_size'])
        dst = out_dir / f'{i:08d}.jpg'
        cv2.imwrite(str(dst), cropped, [cv2.IMWRITE_JPEG_QUALITY, 75])
        rows.append({'image_path': str(dst), 'label': int(row['label'])})
    return pd.DataFrame(rows)

# For first run, set limit=1000 for smoke; remove limit for full run.
train_crop_df = process_split(train_df, 'train', limit=None)
test_crop_df = process_split(test_df, 'test', limit=None)
# Split train into train+val
val_size = int(len(train_crop_df) * CONFIG['val_ratio'])
val_idx = np.random.RandomState(SEED).choice(len(train_crop_df), val_size, replace=False)
mask = np.zeros(len(train_crop_df), dtype=bool); mask[val_idx] = True
val_crop_df = train_crop_df[mask].reset_index(drop=True)
train_crop_df = train_crop_df[~mask].reset_index(drop=True)
print(f'Train: {len(train_crop_df)} | Val: {len(val_crop_df)} | Test: {len(test_crop_df)}')
train_crop_df.to_json(Path(CONFIG['cropped_dir']) / 'manifest_train.json', orient='records')
val_crop_df.to_json(Path(CONFIG['cropped_dir']) / 'manifest_val.json', orient='records')
test_crop_df.to_json(Path(CONFIG['cropped_dir']) / 'manifest_test.json', orient='records')

In [ ]:
def show_crop_comparison(df, n=3):
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    samples = df.sample(n, random_state=SEED)
    for row_idx, (_, row) in enumerate(samples.iterrows()):
        full = cv2.imread(str(ROOT / row['image_path']))
        bbox = load_bbox_for_image(row['image_path'], full.shape) if full is not None else None
        if full is None or bbox is None: continue
        x1, y1, x2, y2 = bbox
        # Original with bbox overlay
        vis = full.copy(); cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,0), 3)
        # Tight crop (baseline: margin 0.15)
        baseline = expand_and_pad_crop(full, bbox, margin=0.15, out_size=112)
        # Context crop (1.8x padded square 224)
        context = expand_and_pad_crop(full, bbox, margin=CONFIG['context_margin_ratio'], out_size=224)
        for col_idx, (img, title) in enumerate([
            (vis, 'Original + bbox'),
            (baseline, 'Baseline (bbox×1.3, 112)'),
            (context, f'Context (bbox×{1 + 2*CONFIG["context_margin_ratio"]:.1f}, 224, padded)'),
        ]):
            axes[row_idx, col_idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[row_idx, col_idx].set_title(f'{title}\nlabel={row["label"]}')
            axes[row_idx, col_idx].axis('off')
    plt.tight_layout()
    plt.savefig(Path(CONFIG['output_dir']) / 'viz2_crop_comparison.png', dpi=80)
    plt.show()

show_crop_comparison(train_df, n=3)

In [ ]:
class FASContextDataset(Dataset):
    def __init__(self, manifest_path, train=True):
        self.df = pd.read_json(manifest_path)
        self.train = train
        if train:
            self.tf = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.2),
                transforms.RandomGrayscale(p=0.05),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Simulate JPEG compression with probability 0.3 during training
        if self.train and random.random() < 0.3:
            q = random.randint(40, 90)
            ok, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, q])
            if ok:
                img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.tf(img), int(row['label'])

train_ds = FASContextDataset(Path(CONFIG['cropped_dir']) / 'manifest_train.json', train=True)
val_ds = FASContextDataset(Path(CONFIG['cropped_dir']) / 'manifest_val.json', train=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

In [ ]:
def show_augmentations(dataset, n_samples=4, n_augs=6):
    fig, axes = plt.subplots(n_samples, n_augs, figsize=(2*n_augs, 2*n_samples))
    indices = np.random.RandomState(SEED).choice(len(dataset), n_samples, replace=False)
    inv_norm = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225])
    for r, idx in enumerate(indices):
        label = dataset.df.iloc[idx]['label']
        for c in range(n_augs):
            tensor_img, _ = dataset[idx]
            np_img = inv_norm(tensor_img).permute(1, 2, 0).clamp(0, 1).numpy()
            axes[r, c].imshow(np_img)
            axes[r, c].axis('off')
            if c == 0:
                axes[r, c].set_ylabel(f'label={label}', fontsize=10)
    plt.suptitle('Same image, 6 augmented variants per row', y=1.01)
    plt.tight_layout()
    plt.savefig(Path(CONFIG['output_dir']) / 'viz3_augmentations.png', dpi=80)
    plt.show()

show_augmentations(train_ds)

In [ ]:
def build_model(num_classes=2, pretrained=True):
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

model = build_model().to(DEVICE)

# Class weights
class_counts = train_ds.df['label'].value_counts().sort_index().values
class_weights = torch.tensor(class_counts.sum() / (2.0 * class_counts), dtype=torch.float32).to(DEVICE)
print('Class weights:', class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
scaler = GradScaler()
print('Model + optimizer ready')

In [ ]:
# === Helpers: per-best artifact export + resumable checkpoint ===

def export_deployment_artifacts(model, val_probs, val_labs, val_acer, epoch_idx, history):
    """Save scripted model, threshold sweep, run_summary.json after each new best.
    Ensures all backend-deployable artifacts exist even if Kaggle kills training mid-run.
    """
    out = Path(CONFIG['output_dir'])

    cpu_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    cpu_model = build_model(num_classes=2, pretrained=False)
    cpu_model.load_state_dict(cpu_state)
    cpu_model.eval()
    example = torch.randn(1, 3, 224, 224)
    scripted = torch.jit.trace(cpu_model, example)
    torch.jit.save(scripted, out / 'mobilenetv2_context_scripted.pt')

    thresholds = np.arange(0.05, 0.96, 0.05)
    sweep = []
    for t in thresholds:
        apcer, bpcer, acer = compute_acer(val_probs, val_labs, threshold=t)
        sweep.append({'threshold': float(t), 'apcer': apcer, 'bpcer': bpcer, 'acer': acer})
    sweep_df = pd.DataFrame(sweep)
    sweep_df.to_csv(out / 'threshold_metrics.csv', index=False)
    best_t_row = sweep_df.loc[sweep_df['acer'].idxmin()]

    with open(out / 'run_summary.json', 'w') as fp:
        json.dump({
            'best_acer': float(val_acer),
            'best_epoch': int(epoch_idx + 1),
            'best_threshold': float(best_t_row['threshold']),
            'image_size': 224,
            'preprocessing': 'imagenet_norm',
            'scripted_checkpoint': str(out / 'mobilenetv2_context_scripted.pt'),
            'best_checkpoint': str(out / 'best_model.pt'),
            'backbone': 'mobilenet_v2_imagenet1k_v1',
            'context_margin_ratio': CONFIG['context_margin_ratio'],
        }, fp, indent=2)

    with open(out / 'history.json', 'w') as fp:
        json.dump(history, fp, indent=2)

    return float(best_t_row['threshold'])


def save_resume_checkpoint(model, optimizer, scheduler, scaler, epoch_idx, history, best_acer):
    torch.save({
        'epoch': epoch_idx + 1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'history': history,
        'best_acer': best_acer,
    }, Path(CONFIG['output_dir']) / 'resume_checkpoint.pt')


# === Optional resume from a prior interrupted run ===
history = {'train_loss': [], 'val_acer': [], 'val_apcer': [], 'val_bpcer': []}
best_acer = float('inf'); best_epoch = -1
no_improve = 0
start_epoch = 0

resume_path = Path(CONFIG['output_dir']) / 'resume_checkpoint.pt'
if CONFIG.get('resume', False) and resume_path.exists():
    print(f'Resuming from {resume_path}')
    ckpt = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    start_epoch = ckpt['epoch']
    history = ckpt['history']
    best_acer = ckpt['best_acer']
    print(f'Resumed at epoch {start_epoch}; best_acer so far={best_acer:.4f}')

# === Training loop with early stopping + per-best artifact export ===
for epoch in range(start_epoch, CONFIG['epochs']):
    model.train()
    running_loss = 0.0; n = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["epochs"]}')
    for imgs, y in pbar:
        imgs = imgs.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out = model(imgs); loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running_loss += loss.item() * imgs.size(0); n += imgs.size(0)
        pbar.set_postfix(loss=f'{running_loss/n:.4f}')
    scheduler.step()
    train_loss = running_loss / n

    val_acer, val_apcer, val_bpcer, val_probs, val_labs = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    history['val_acer'].append(val_acer)
    history['val_apcer'].append(val_apcer)
    history['val_bpcer'].append(val_bpcer)
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f} val_ACER={val_acer:.4f} APCER={val_apcer:.4f} BPCER={val_bpcer:.4f}')

    if val_acer < best_acer:
        best_acer = val_acer; best_epoch = epoch; no_improve = 0
        torch.save(model.state_dict(), Path(CONFIG['output_dir']) / 'best_model.pt')
        best_t = export_deployment_artifacts(model, val_probs, val_labs, val_acer, epoch, history)
        print(f'  NEW BEST: saved best_model.pt + scripted + threshold_metrics.csv + run_summary.json (best_threshold={best_t:.2f})')
    else:
        no_improve += 1
        print(f'  no improvement ({no_improve}/{CONFIG["patience"]})')

    save_resume_checkpoint(model, optimizer, scheduler, scaler, epoch, history, best_acer)

    if no_improve >= CONFIG['patience']:
        print(f'Early stop: no val improvement for {CONFIG["patience"]} epochs')
        break

print()
print(f'Training done. Best epoch: {best_epoch+1} | Best val ACER: {best_acer:.4f}')
print(f'Deployment artifacts already saved to {CONFIG["output_dir"]} during training.')


In [ ]:
# === Final verification — artifacts were already exported during training ===
out = Path(CONFIG['output_dir'])

print('Artifacts in output dir:')
for name in ['best_model.pt', 'mobilenetv2_context_scripted.pt', 'run_summary.json',
             'threshold_metrics.csv', 'history.json', 'resume_checkpoint.pt']:
    p = out / name
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f'  OK  {name}  ({size_mb:.2f} MB)')
    else:
        print(f'  MISSING  {name}')

with open(out / 'run_summary.json') as fp:
    summary = json.load(fp)
print()
print('Run summary:')
print(json.dumps(summary, indent=2))

# Reload val predictions for plots
sweep_df = pd.read_csv(out / 'threshold_metrics.csv')
best_t_row = sweep_df.loc[sweep_df['acer'].idxmin()]
print(f'\nBest threshold from sweep: {best_t_row["threshold"]:.2f} (ACER={best_t_row["acer"]:.4f})')

# ACER curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df['threshold'], sweep_df['apcer'], label='APCER (spoof->live)')
ax.plot(sweep_df['threshold'], sweep_df['bpcer'], label='BPCER (live->spoof)')
ax.plot(sweep_df['threshold'], sweep_df['acer'], label='ACER', linewidth=2)
ax.axvline(best_t_row['threshold'], color='r', linestyle='--', label=f'best t={best_t_row["threshold"]:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Error rate'); ax.legend(); ax.grid()
plt.tight_layout()
plt.savefig(out / 'threshold_sweep.png', dpi=80)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

out = Path(CONFIG['output_dir'])

# Reload best model + re-evaluate on val to get fresh probs (safe after any restart)
model_eval = build_model(num_classes=2, pretrained=False)
model_eval.load_state_dict(torch.load(out / 'best_model.pt', map_location=DEVICE))
model_eval = model_eval.to(DEVICE); model_eval.eval()
val_acer_r, val_apcer_r, val_bpcer_r, val_probs_r, val_labs_r = evaluate(model_eval, val_loader)
print(f'Reloaded best model val_ACER={val_acer_r:.4f}')

with open(out / 'run_summary.json') as fp:
    best_t = float(json.load(fp)['best_threshold'])

preds = (np.array(val_probs_r) >= best_t).astype(int)
labs = np.array(val_labs_r)
cm = confusion_matrix(labs, preds, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['spoof', 'live'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['spoof', 'live'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
ax.set_title(f'Val Confusion Matrix @ t={best_t:.2f}')
plt.tight_layout()
plt.savefig(out / 'viz4_confusion_matrix.png', dpi=80)
plt.show()

# Hard examples
probs_arr = np.array(val_probs_r); labs_arr = np.array(val_labs_r)
live_as_spoof_idx = np.where((labs_arr == 1) & (probs_arr < best_t))[0]
spoof_as_live_idx = np.where((labs_arr == 0) & (probs_arr >= best_t))[0]
live_as_spoof_idx = live_as_spoof_idx[np.argsort(probs_arr[live_as_spoof_idx])[:3]]
spoof_as_live_idx = spoof_as_live_idx[np.argsort(-probs_arr[spoof_as_live_idx])[:3]]

if len(live_as_spoof_idx) > 0 or len(spoof_as_live_idx) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for i, idx in enumerate(live_as_spoof_idx[:3]):
        img_path = val_ds.df.iloc[idx]['image_path']
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img); axes[0, i].axis('off')
        axes[0, i].set_title(f'LIVE -> SPOOF\nlive_prob={probs_arr[idx]:.3f}')
    for i, idx in enumerate(spoof_as_live_idx[:3]):
        img_path = val_ds.df.iloc[idx]['image_path']
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        axes[1, i].imshow(img); axes[1, i].axis('off')
        axes[1, i].set_title(f'SPOOF -> LIVE\nlive_prob={probs_arr[idx]:.3f}')
    plt.tight_layout()
    plt.savefig(out / 'viz4_hard_examples.png', dpi=80)
    plt.show()
else:
    print('No hard examples — model classifies all val samples correctly at this threshold.')
